In [ ]:
puts `ls -l`

In [6]:
puts `head -3 ./raw_data/bv-kg-20250225.large`

source_1	id_1	type_1	name_1	source_2	id_2	type_2	name_2	score	url
UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	HP:human_phenotype	HP:0001263	Human Phenotype	Global developmental delay	0.0501869	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Human%20Phenotype%7CGlobal%20developmental%20delay%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D
UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	HP:human_phenotype	HP:0001249	Human Phenotype	Intellectual disability	0.0438494	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Human%20Phenotype%7CIntellectual%20disability%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D


In [7]:
puts `head -3 ./maps/2025-biovista-genes.map`

bv_geneid,bv_label,geneid,protein,recommended_full,taxon
11758,GPx,http://purl.uniprot.org/geneid/11758,http://purl.uniprot.org/uniprot/O08709,Peroxiredoxin-6,http://purl.uniprot.org/taxonomy/10090
1213,HC,http://purl.uniprot.org/geneid/1213,http://purl.uniprot.org/uniprot/Q00610,Clathrin heavy chain 1,http://purl.uniprot.org/taxonomy/9606


In [8]:
require 'linkeddata'
require 'rdf/nquads'
require 'csv'
require 'net/http'
require 'json'
require 'uri'

graphing_errors = File.open('./graph/2026_biovista_gene-phenotype-errors.txt', 'w')

SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS      = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

gene_mappings = CSV.read('./maps/2025-biovista-genes.map', headers: true)

$hpo_cache = {}

def get_hpo_label(hp_code)
  return $hpo_cache[hp_code] if $hpo_cache.key?(hp_code)
  obo_id  = hp_code.start_with?('HP_') ? hp_code : hp_code.sub('HP:', 'HP_')
  iri     = "http://purl.obolibrary.org/obo/#{obo_id}"
  encoded = URI.encode_www_form_component(URI.encode_www_form_component(iri))
  uri     = URI("https://www.ebi.ac.uk/ols4/api/ontologies/hp/terms/#{encoded}")
  resp    = Net::HTTP.get_response(uri)
  result  = resp.is_a?(Net::HTTPSuccess) ? (JSON.parse(resp.body)['label'] || "HPO term #{obo_id}") : "HPO term #{obo_id}"
  $hpo_cache[hp_code] = result
rescue
  $hpo_cache[hp_code] = "HPO term #{hp_code}"
end

failures     = {}
record_count = 0

# Refresh output file
File.open('./graph/2026_biovista_gene-phenotype.nq.large', 'w').close

CSV.foreach('./raw_data/bv-kg-20250225.large', col_sep: "\t", quote_char: '"',
            liberal_parsing: true, headers: true) do |row|

  next unless (row['type_1'] == 'Gene'           && row['type_2'] == 'Human Phenotype') ||
              (row['type_1'] == 'Human Phenotype' && row['type_2'] == 'Gene')

  if row['type_1'] == 'Gene'
    gene_id  = row['id_1']
    pheno_id = row['id_2']
  else
    gene_id  = row['id_2']
    pheno_id = row['id_1']
  end

  next if gene_id =~ /[A-Z]/   # skip MeSH category IDs (contain letters)

  score    = row['score']
  evidence = row['url']

  gene = gene_mappings.find { |d| d['bv_geneid'] == gene_id }
  unless gene
    unless failures[gene_id]
      failures[gene_id] = true
      graphing_errors.write "gene lookup failed #{gene_id}\n"
    end
    next
  end

  unless pheno_id =~ /HP:\d+/
    graphing_errors.write "unrecognised phenotype ID #{pheno_id}\n"
    next
  end

  hp_num  = pheno_id.sub(':', '_')   # 'HP:0001263' -> 'HP_0001263'
  hpo_uri = RDF::URI.new("http://purl.obolibrary.org/obo/#{hp_num}")

  gene_uri           = RDF::URI.new(gene['geneid'])
  gene_type          = RDF::URI.new('http://edamontology.org/data_1027')
  gene_core_type     = RDF::URI.new('https://w3id.org/biolink/vocab/Gene')
  biovista_gene_label = RDF::Literal.new(gene['recommended_full'])

  protein_uri            = RDF::URI.new(gene['protein'])
  protein_type           = RDF::URI.new('http://edamontology.org/data_2291')
  protein_core_type      = RDF::URI.new('https://w3id.org/biolink/vocab/Protein')
  biovista_protein_label = RDF::Literal.new(gene['recommended_full'])

  taxon = RDF::URI.new(gene['taxon'])

  hpo_type      = RDF::URI.new('http://edamontology.org/data_3275')
  hpo_core_type = RDF::URI.new('https://w3id.org/biolink/vocab/Phenotype')
  hpo_label     = RDF::Literal.new(get_hpo_label(hp_num))

  context_uri     = RDF::URI.new("urn:simpathic:context:bv_#{gene_id}_#{hp_num}")
  general_context = RDF::URI.new('urn:simpathic:context:all_metadata')

  graph = RDF::Repository.new

  # Gene <-> Phenotype
  graph << RDF::Statement.new(gene_uri, SIMPATHIC['associated-with'], hpo_uri,  graph_name: context_uri)
  graph << RDF::Statement.new(hpo_uri,  SIMPATHIC['associated-with'], gene_uri, graph_name: context_uri)

  # Protein <-> Phenotype
  graph << RDF::Statement.new(protein_uri, SIMPATHIC['associated-with'], hpo_uri,     graph_name: context_uri)
  graph << RDF::Statement.new(hpo_uri,     SIMPATHIC['associated-with'], protein_uri, graph_name: context_uri)

  # Gene entity
  graph << RDF::Statement.new(gene_uri,       RDFS.label,               biovista_gene_label,               graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,       RDF.type,                 gene_type,                         graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,       RDF.type,                 gene_core_type,                    graph_name: context_uri)
  graph << RDF::Statement.new(gene_type,      RDFS.label,               RDF::Literal.new('NCBI Gene'),     graph_name: context_uri)
  graph << RDF::Statement.new(gene_core_type, RDFS.label,               RDF::Literal.new('Gene'),          graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,       SIMPATHIC['original-id'], RDF::Literal.new(gene_id),         graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,       SIMPATHIC['in-taxon'],    taxon,                             graph_name: context_uri)

  # Protein entity
  graph << RDF::Statement.new(protein_uri,       RDFS.label,               biovista_protein_label,            graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,       RDF.type,                 protein_type,                      graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,       RDF.type,                 protein_core_type,                 graph_name: context_uri)
  graph << RDF::Statement.new(protein_type,      RDFS.label,               RDF::Literal.new('UniProt'),       graph_name: context_uri)
  graph << RDF::Statement.new(protein_core_type, RDFS.label,               RDF::Literal.new('Protein'),       graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,       SIMPATHIC['original-id'], RDF::Literal.new(gene_id),         graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,       SIMPATHIC['in-taxon'],    taxon,                             graph_name: context_uri)

  # HPO entity
  graph << RDF::Statement.new(hpo_uri,       RDFS.label,               hpo_label,                             graph_name: context_uri)
  graph << RDF::Statement.new(hpo_uri,       RDF.type,                 hpo_type,                              graph_name: context_uri)
  graph << RDF::Statement.new(hpo_uri,       RDF.type,                 hpo_core_type,                         graph_name: context_uri)
  graph << RDF::Statement.new(hpo_type,      RDFS.label,               RDF::Literal.new('HPO Ontology Term'), graph_name: context_uri)
  graph << RDF::Statement.new(hpo_core_type, RDFS.label,               RDF::Literal.new('Phenotype'),         graph_name: context_uri)
  graph << RDF::Statement.new(hpo_uri,       SIMPATHIC['original-id'], RDF::Literal.new(hp_num),              graph_name: context_uri)

  # Context metadata
  graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'],      RDF::Literal.new('Biovista'),        graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'],         RDF::URI.new(evidence),              graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['score'],            RDF::Literal.new(score),             graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'], RDF::Literal.new('ASSOCIATED_WITH'),  graph_name: general_context)

  File.open('./graph/2026_biovista_gene-phenotype.nq.large', 'a') do |f|
    RDF::Writer.for(:nquads).new(f) do |writer|
      writer << graph
    end
  end

  record_count += 1
  warn "#{record_count} records processed" if (record_count % 500).zero?
end

graphing_errors.close
puts "Done -- #{record_count} gene-phenotype associations written"
puts "See ./graph/2026_biovista_gene-phenotype-errors.txt for failures"

(irb):9: warning: already initialized constant Object::SIMPATHIC
(irb):6: warning: previous definition of SIMPATHIC was here
(irb):10: warning: already initialized constant Object::RDFS
(irb):7: warning: previous definition of RDFS was here
500 records processed
1000 records processed
1500 records processed
2000 records processed
2500 records processed
3000 records processed
3500 records processed
4000 records processed
4500 records processed
5000 records processed
5500 records processed
6000 records processed
6500 records processed
7000 records processed
7500 records processed
8000 records processed
8500 records processed
9000 records processed
9500 records processed
10000 records processed
10500 records processed
11000 records processed
11500 records processed
12000 records processed
12500 records processed
13000 records processed
13500 records processed
14000 records processed
14500 records processed
15000 records processed
15500 records processed
16000 records processed
16500 recor

Done -- 92105 gene-phenotype associations written
See ./graph/2026_biovista_gene-phenotype-errors.txt for failures
